# RFSoC 4×2 · Method B — RX + TCP + AI（B 板）
B 板專用，整合三份來源：
- **FPGA release**（GitHub `OFDM_two_board_4subcarriers_16QAM_webcam`）：`ofdm_qam16.bit/.hwh/manifest.json`，4 子載波 16-QAM、BRAM 區塊、CRC 與 ACK 重傳。
- **Method B TX/RX 分板版**：`on_decoded` hook、RAM buffer、TCP Gray8 轉送、PSNR、`duration_s`。
- **1125 AI 套件（精簡）**：SSD-MobileNet V1 INT8 TFLite 的 spawn worker，透過 `on_decoded` 即時接上 RX 解碼影格。

AI 有兩條路，可同時開：TCP 把 Gray8 送到 Rubik Pi／PC 做 AI（系統圖的主路徑）；或 `AI_ENABLED=True` 在 B 板 PS 上推論（對照組，預設關閉）。
單板預設用測試影像跑 TX/RX BRAM、解碼、callback 與錄影。雙板請改成 `MODE = "live"`，在 B 出現 `RX_WAITING_FOR_PILOT` 後啟動 A。從本資料夾啟動 Jupyter（`BITFILE = Path.cwd() / ...`），更新檔案後重啟 kernel。

## 1. RX configuration

In [ ]:
from pathlib import Path
import json, hashlib, re
from collections import deque, OrderedDict
import threading
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from webcam_benchmark import run_single_board
from webcam_live import run_live
from webcam_stream import WIDTH, HEIGHT, JPEG_BUDGET
from webcam_latency import show_latency

BUNDLE = Path.cwd()
BITFILE = BUNDLE / "ofdm_qam16.bit"
OUTPUT_ROOT = BUNDLE / "received_live_webcam"
FPS = 30.0
AMPLITUDE = 4096
LIVE_PREVIEW = True
RECORDING_CODEC = "auto"
PAIR_SESSION = "test_001"  # paired export only; use a NEW matching ID on both boards
assert re.fullmatch(r"[A-Za-z0-9_-]+", PAIR_SESSION)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

ROLE = "receive"
MODE = "single_board"  # 單板 BRAM 測試。雙板 RF 設為 "live"
BENCH_CAMERA = None    # 固定測試圖。要用 USB 攝影機時填 0
BENCH_BLOCKS = 3
BENCH_FRAMES = 3
if MODE not in ("single_board", "live"):
    raise ValueError("MODE must be single_board or live")
MAX_RAM_FRAMES = 120  # 0 = 不保留解碼影像
EXPORT_RX_JPEG = False  # 配對 PSNR 才開啟；同步磁碟 I/O 會影響效能
RUN_PAIRED_PSNR = False  # 將 TX reference 複製到 B 後，才執行配對 Cell
MAX_LOG_FRAMES = 1000
assert MAX_RAM_FRAMES >= 0

# Ethernet TCP destination — set to the server computer's IP before running.
TCP_ENABLED = False    # 接上 Rubik Pi／PC 並備妥 transport.protocol 後才開啟
TCP_HOST = "127.0.0.1"  # SAME BOARD ONLY. Replace with PC LAN IP, e.g. 192.168.1.100
TCP_PORT = 9001
TCP_QUEUE_FRAMES = 30    # bounded, drop oldest unsent frame on overload
TCP_TIMEOUT_S = 3.0     # connect and sendall timeout
TCP_DRAIN_S = 5.0       # max graceful drain time after OFDM stops
assert TCP_QUEUE_FRAMES > 0 and TCP_TIMEOUT_S > 0 and TCP_DRAIN_S >= 0

# PS-side AI on board B (optional). Model/labels/config are NOT bundled; see docs/MODEL.md.
AI_ENABLED = False      # True: start the SSD worker before RF; frames come from on_decoded
AI_MOCK = False         # True: data-path test only, empty detections marked mock=true
AI_FPS = 2.0            # AI sampling rate; independent of RF / recording FPS
AI_CONFIG = BUNDLE / "config" / "detector.json"
AI_RESULTS_ROOT = BUNDLE / "results"
assert AI_FPS > 0


## 2. TCP sender — RAM Queue and persistent connection
沿用原 `encode_frame(640, 480, frame_id, time_ns(), image.tobytes())`。不使用 `LocalSource`、`cv2.imread` 或 MP4 回讀。TCP 連線在執行 RF 前建立；若連不上就先報錯。連線中斷後停止 TCP 並統計錯誤，OFDM / 錄影繼續。佇列滿時丟最舊尚未傳送的幀，避免慢速 TCP 阻塞錄影 callback。

In [ ]:
import socket
from time import time_ns

class RamTcpSender:
    """One connection, bounded RAM FIFO, independent TCP sender thread."""
    def __init__(self, host, port, encode_frame, max_frames=30, timeout=3.0):
        self.host, self.port = host, int(port)
        self.encode_frame = encode_frame
        self.max_frames, self.timeout = int(max_frames), float(timeout)
        if self.max_frames < 1 or self.timeout <= 0:
            raise ValueError("Invalid TCP queue capacity or timeout")
        self.pending = deque()
        self.cv = threading.Condition()
        self.closing = False
        self.error = None
        self.sock = self.worker = None
        self.counts = dict(submitted=0, sent=0, dropped_overflow=0,
                           dropped_shutdown=0, dropped_error=0,
                           failed_inflight=0, rejected=0)

    def start(self):
        # Fail before starting RF if the configured server cannot be reached.
        self.sock = socket.create_connection((self.host, self.port), timeout=self.timeout)
        try:
            self.sock.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
            self.sock.settimeout(self.timeout)
            self.worker = threading.Thread(target=self._run, name="RAM-TCP", daemon=True)
            self.worker.start()
        except BaseException:
            self.sock.close()
            raise
        return self

    def submit(self, frame_id, image):
        # Recording produces a fresh gray array for each frame. Keep a strong
        # reference (even after the frame is evicted from received_frames).
        # Do not mutate the array; get_received_frame() returns a copy.
        if image.shape != (480, 640) or image.dtype != np.uint8:
            raise ValueError("TCP requires a 640x480 Gray8 image")
        with self.cv:
            if self.closing or self.error is not None:
                self.counts["rejected"] += 1
                return False
            self.counts["submitted"] += 1
            if len(self.pending) >= self.max_frames:
                self.pending.popleft()
                self.counts["dropped_overflow"] += 1
            self.pending.append((int(frame_id), image))
            self.cv.notify()
            return True

    def _run(self):
        try:
            while True:
                with self.cv:
                    self.cv.wait_for(lambda: self.pending or self.closing)
                    if not self.pending:
                        return
                    frame_id, image = self.pending.popleft()
                try:
                    # Same signature, layout and sender wall-clock timestamp
                    # as mock_rfsoc_sender.py; raw grayscale, not JPEG.
                    packet = self.encode_frame(640, 480, frame_id, time_ns(), image.tobytes(order="C"))
                    self.sock.sendall(packet)
                except Exception as exc:
                    with self.cv:
                        self.error = repr(exc)
                        self.counts["failed_inflight"] += 1
                        self.counts["dropped_error"] += len(self.pending)
                        self.pending.clear()
                        self.closing = True
                    return
                with self.cv:
                    self.counts["sent"] += 1
        finally:
            self.sock.close()

    def finish(self, drain_seconds=5.0):
        # No new submissions; drain pending frames up to the requested time.
        with self.cv:
            self.closing = True
            self.cv.notify_all()
        if self.worker is not None:
            self.worker.join(timeout=drain_seconds)
            if self.worker.is_alive():
                with self.cv:
                    self.counts["dropped_shutdown"] += len(self.pending)
                    self.pending.clear()
                    self.cv.notify_all()
                try:
                    self.sock.shutdown(socket.SHUT_RDWR)
                except OSError:
                    pass
                self.worker.join(timeout=self.timeout + 0.5)
        return self.status()

    def status(self):
        with self.cv:
            return dict(self.counts, queued=len(self.pending), error=self.error,
                        worker_alive=bool(self.worker and self.worker.is_alive()),
                        host=self.host, port=self.port)

# Created in the Run RX cell, not on import.
tcp_sender = None


## 3. AI worker（B 板 PS，可選）
`AIWorker` 是獨立 spawn process，只有一個 shared-memory slot。`submit()` 只做格式檢查與 307,200-byte copy；鎖忙立刻回 `False`，從不等待推論或 I/O，因此不會反壓 RF／錄影。worker 依 `AI_FPS` 取最新幀推論，結果寫入 `results/live_<時間>/detections.jsonl` 與 `latest.json`。
`frame_id` 用 A 板的相機影格編號（RFJ2 entry index），與 PSNR／TCP 同一編號。`source_timestamp_ns` 傳 `None`：A 的取像時間是 A 端相對秒數，兩板時鐘未同步，不可當成 B 的時間。

In [ ]:
import sys, datetime
if str(BUNDLE) not in sys.path:
    sys.path.insert(0, str(BUNDLE))   # spawn child re-imports ai.ai_worker from this folder
from ai.ai_worker import AIWorker

# Created in the Run RX cell, not on import.
ai_worker = None
ai_run_dir = None

def start_ai_worker():
    if not AI_MOCK and not AI_CONFIG.is_file():
        raise FileNotFoundError(f"{AI_CONFIG} missing: copy config/detector.example.json, add model/labels, "
                                "confirm the contract (docs/MODEL.md) or set AI_MOCK=True")
    run_dir = AI_RESULTS_ROOT / ("live_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    worker = AIWorker(config=None if AI_MOCK else AI_CONFIG, shape=(HEIGHT, WIDTH),
                      fps=AI_FPS, out=run_dir, mock=AI_MOCK).start()  # model load finishes before RF
    return worker, run_dir

## 4. RX live RAM buffer、TCP 與 AI callback
直接接收 Recording.append() 已解碼的 gray，不重複 imdecode。使用原 RF frame ID 作 key，只保留最近 N 張。TCP Queue 與 RAM Buffer 共用同一張灰階陣列的 reference；即使 RAM Buffer 淘汰該幀，Queue 仍可持有它直到傳送。AI worker 則複製一份到 shared memory，不保留 reference。

In [ ]:
received_frames = OrderedDict()
rx_lock = threading.Lock()
rx_log = deque(maxlen=MAX_LOG_FRAMES)
rx_errors = deque(maxlen=20)
RX_EXPORT_DIR = OUTPUT_ROOT / "pair_rx" / PAIR_SESSION
REFERENCE_DIR = OUTPUT_ROOT / "pair_reference" / PAIR_SESSION
if EXPORT_RX_JPEG:
    RX_EXPORT_DIR.mkdir(parents=True, exist_ok=False)

def on_decoded(frame_id, gray, jpeg):
    try:
        with rx_lock:
            if MAX_RAM_FRAMES:
                received_frames[frame_id] = gray
                received_frames.move_to_end(frame_id)
                while len(received_frames) > MAX_RAM_FRAMES:
                    received_frames.popitem(last=False)
            rx_log.append(dict(frame_id=int(frame_id), jpeg_bytes=len(jpeg)))
        if tcp_sender is not None:
            tcp_sender.submit(frame_id, gray)
        if ai_worker is not None:
            # Non-blocking; False only means AI skipped this frame (busy/unavailable).
            ai_worker.submit(gray, int(frame_id), source_timestamp_ns=None)
        if EXPORT_RX_JPEG:
            (RX_EXPORT_DIR / f"{frame_id}.jpg").write_bytes(jpeg)
    except Exception as exc:
        rx_errors.append(repr(exc))

def get_received_frame(frame_id=None):
    with rx_lock:
        if not received_frames:
            return None
        key = next(reversed(received_frames)) if frame_id is None else frame_id
        frame = received_frames.get(key)
        return None if frame is None else frame.copy()
# Alias if you prefer frames[index] instead of received_frames[index].
frames = received_frames


## 5. Run RX — OFDM、解包、即時顯示、背景錄影、RAM、TCP 與 AI
單板只需執行本 Notebook。要轉送影像時才開 TCP server，設定 TCP_HOST 並開啟 TCP_ENABLED。正常 EOS 或中斷後會關閉 TCP 連線並輸出傳送統計。AI 在 RF 之前啟動（模型載入失敗就不會開始等 pilot）；`run_live` 返回時錄影執行緒已結束，不會再有新影格，之後才關 AI worker。

In [ ]:
tcp_sender = None
tcp_summary = None
ai_worker = None
ai_summary = None
try:
    if AI_ENABLED:
        ai_worker, ai_run_dir = start_ai_worker()
        print(f"AI worker ready ({'MOCK' if AI_MOCK else 'model'}), results: {ai_run_dir}", flush=True)
    if TCP_ENABLED:
        # Import the original protocol, NOT an invented replacement header.
        from transport.protocol import encode_frame
        tcp_sender = RamTcpSender(TCP_HOST, TCP_PORT, encode_frame,
                                  TCP_QUEUE_FRAMES, TCP_TIMEOUT_S).start()
        print(f"TCP connected to {TCP_HOST}:{TCP_PORT}", flush=True)
    if MODE == "single_board":
        summary = run_single_board(BITFILE, OUTPUT_ROOT, camera=BENCH_CAMERA, fps=FPS,
                                   blocks=BENCH_BLOCKS, frames=BENCH_FRAMES,
                                   recording_codec=RECORDING_CODEC, on_decoded=on_decoded)
    else:
        summary = run_live(ROLE, BITFILE, OUTPUT_ROOT, fps=FPS,
                           amplitude=AMPLITUDE, show_preview=LIVE_PREVIEW,
                           recording_codec=RECORDING_CODEC, on_decoded=on_decoded)
finally:
    if tcp_sender is not None:
        tcp_summary = tcp_sender.finish(TCP_DRAIN_S)
        print("TCP summary:", json.dumps(tcp_summary, indent=2), flush=True)
    if ai_worker is not None:
        worker, ai_worker = ai_worker, None   # detach the producer before close
        worker.close()
        ai_summary = json.loads((ai_run_dir / "worker_summary.json").read_text())
        print("AI summary:", json.dumps(ai_summary, indent=2), flush=True)
if tcp_summary is not None:
    summary["tcp"] = tcp_summary  # notebook summary; original backend JSON is unchanged
if ai_summary is not None:
    summary["ai"] = dict(ai_summary, run_dir=str(ai_run_dir), mock=AI_MOCK)

## 6. RX metrics、RAM / MP4 預覽與 AI 結果

In [ ]:
if summary.get("latency_report"):
    show_latency(summary["latency_report"])
print(json.dumps(summary, indent=2, default=str, ensure_ascii=False))
hist = summary.get("frames_per_block_histogram", {})
if hist:
    x = sorted(map(int, hist))
    plt.bar(x, [hist.get(k, hist.get(str(k))) for k in x])
    plt.xlabel("JPEG frames per data block")
    plt.ylabel("Blocks")
    plt.xticks(x)
    plt.show()

print("RAM frames:", len(received_frames),
      "MiB:", sum(a.nbytes for a in received_frames.values()) / 2**20)
print("Analysis errors:", list(rx_errors))
(OUTPUT_ROOT / f"rx_{PAIR_SESSION}_metrics.json").write_text(
    json.dumps(list(rx_log), indent=2), encoding="utf-8")
image = get_received_frame()
if image is not None:
    plt.imshow(image, cmap="gray", vmin=0, vmax=255)
    plt.axis("off"); plt.show()
if summary.get("mp4"):
    from IPython.display import Video
    p = Path(summary["mp4"])
    print("MP4:", p)
    if p.is_file():
        try:
            display(Video(url=p.resolve().relative_to(BUNDLE.resolve()).as_posix(),
                          embed=False, width=640))
        except ValueError:
            print("Use the Jupyter file browser to open the recording.")

In [ ]:
# AI results: latency summary + latest detections drawn on the matching RAM frame
if summary.get("ai"):
    import subprocess
    run_dir = Path(summary["ai"]["run_dir"])
    subprocess.run([sys.executable, "-m", "tools.summarize_results", str(run_dir)], cwd=BUNDLE, check=False)
    if (run_dir / "worker_error.json").exists():
        print((run_dir / "worker_error.json").read_text())
    latest_path = run_dir / "latest.json"
    if latest_path.exists():
        latest = json.loads(latest_path.read_text())
        print("Latest AI frame:", latest["frame_id"], "detections:", len(latest["detections"]),
              "pipeline_ms:", round(latest["pipeline_ms"], 2))
        img = get_received_frame(latest["frame_id"])  # None if evicted from the RAM buffer
        if img is not None:
            vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
            for d in latest["detections"]:
                x0, y0, x1, y1 = map(int, d["bbox_xyxy_pixels"])
                cv2.rectangle(vis, (x0, y0), (x1, y1), (0, 255, 0), 2)
                cv2.putText(vis, f'{d["label"]} {d["score"]:.2f}', (x0, max(12, y0 - 4)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
            plt.figure(figsize=(8, 6)); plt.imshow(vis[:, :, ::-1]); plt.axis("off")
            plt.title(f'AI frame {latest["frame_id"]}'); plt.show()

## 7. Optional cross-board PSNR
兩板啟用 export 且 PAIR_SESSION 相同、每次唯一。將 TX `pair_reference/<PAIR_SESSION>` 複製到 B 同名目錄。
以 frame ID 和 JPEG SHA-256 配對，使用原始 JPEG 而非可能重新編碼／補幀的 MP4。
完成傳輸後可將 RUN_PAIRED_PSNR 設為 True 並單獨執行下一格。

In [ ]:
if RUN_PAIRED_PSNR:
    if not REFERENCE_DIR.is_dir() or not RX_EXPORT_DIR.is_dir():
        raise FileNotFoundError("Need matching TX reference and RX JPEG export directories")
    paired = []
    skipped = []
    examples = []
    for p in sorted(RX_EXPORT_DIR.glob("*.jpg"), key=lambda f: int(f.stem)):
        ref = REFERENCE_DIR / (p.stem + ".npz")
        if not ref.is_file():
            skipped.append((p.stem, "missing TX reference")); continue
        jpeg = p.read_bytes()
        with np.load(ref, allow_pickle=False) as z:
            expected = str(z["jpeg_sha256"].item())
            gray = z["gray"]
        if hashlib.sha256(jpeg).hexdigest() != expected:
            skipped.append((p.stem, "JPEG digest mismatch")); continue
        img = cv2.imdecode(np.frombuffer(jpeg, np.uint8), cv2.IMREAD_GRAYSCALE)
        if img is None or img.shape != gray.shape or gray.dtype != np.uint8:
            skipped.append((p.stem, "decode/shape/type mismatch")); continue
        psnr = float(cv2.PSNR(gray, img))
        paired.append(dict(frame_id=int(p.stem), psnr_db=psnr))
        if len(examples) < 3:
            examples.append((p.stem, gray, img, psnr))
    print("Matched:", len(paired), "Skipped:", len(skipped))
    (OUTPUT_ROOT / f"paired_{PAIR_SESSION}.json").write_text(json.dumps(dict(paired=paired, skipped=skipped), indent=2), encoding="utf-8")
    if paired:
        plt.plot([r["frame_id"] for r in paired], [r["psnr_db"] for r in paired]); plt.xlabel("wire frame ID"); plt.ylabel("PSNR (dB)"); plt.show()
        for idx, src, dst, psnr in examples:
            plt.figure(figsize=(10, 4)); plt.imshow(np.hstack((src, dst)), cmap="gray", vmin=0, vmax=255)
            plt.title(f"Frame {idx}: TX original | RX decoded — {psnr:.2f} dB"); plt.axis("off"); plt.show()

## 8. Optional AI replay on the recorded MP4
即時 AI 若影響 RF 指標，可關 `AI_ENABLED`，先錄影再離線推論。replay 不碰 RF/PL；frame_id 為 MP4 內的本地序號（非 A 的影格編號，MP4 可能有補幀）。

In [ ]:
RUN_AI_REPLAY = False
if RUN_AI_REPLAY:
    import subprocess
    video = Path(summary["mp4"])
    replay_dir = AI_RESULTS_ROOT / ("replay_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    cmd = [sys.executable, "-m", "ai.replay", "--video", str(video), "--config", str(AI_CONFIG),
           "--out", str(replay_dir), "--ai-fps", str(AI_FPS)] + (["--mock"] if AI_MOCK else [])
    subprocess.run(cmd, cwd=BUNDLE, check=True)
    subprocess.run([sys.executable, "-m", "tools.summarize_results", str(replay_dir)], cwd=BUNDLE, check=False)

## 9. 注意事項
- 原生預覽直接顯示每個 block 最後一張 JPEG；不等於全部幀的播放率。
- RAM callback 隨背景錄影執行，非 RF DONE 當下；錄影佇列滿時可能反壓接收流程。
- 120 張 640×480 灰階約 35.16 MiB，不含其他 buffer。
- 真正跨板 end-to-end latency 需時鐘同步；原有各板計時保留。
- 不新增 PS block 重傳或定期 pilot calibration，避免破壞雙板 descriptor 與握手。
- TCP 封包為 Gray8 原始像素，不是 JPEG：640×480 = 307,200 bytes/frame（不含 header）。30 fps 約 73.728 Mbit/s payload。
- TCP sender 不限制固定 10 fps，依真實解碼幀送出，可能成批突發；來源 frame ID 可能有缺號。`sent` 代表 sendall 成功，非對端應用程式確認。
- TCP queue 最多保留 30 張待傳影像（另有一張處理中）；傳輸跟不上則丟棄最舊未傳影像，記於 dropped_overflow。TCP 斷線不重連，保留 RF/錄影。
- 原 TCP server 及 transport.protocol.py 未提供，請沿用原專案的協定模組；未驗證真實 server 的 header 與 frame ID 限制。
- 原有 Jupyter 預覽每 block 最後一張 JPEG，不代表所有轉送的 frame；本版不修改 TX 或 RF 協定。
- AI：`AI_ENABLED=True` 需要 `models/ssd_mobilenet_v1_int8.tflite`、`labels/labels.txt`、已確認契約的 `config/detector.json`（`contract_confirmed=true`）與板上相容的 `tflite_runtime`。缺任一項會在 RF 開始前報錯；先用 `AI_MOCK=True` 只驗證資料流。
- AI 與 RX 錄影、TCP 共用 PS 的 4 顆 A53。比較 `AI_ENABLED` 開/關時的 `delivered_fps`、`done_to_release`、`rx_service`、`record_queue_wait` 與 `rx_analysis`；`worker_summary.json` 的 `submit_max_ms` 是 callback 中 AI 的最壞耗時。
- AI `accepted` 不是辨識數；完成數看 `detections.jsonl` 行數。停止時可能丟掉最後一張 pending 幀。
